In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException
import time
from utils import writeJson, readJson
import os
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
import json
import re
from bs4 import BeautifulSoup as soup
import datetime
from datetime import datetime as dt
from tqdm import tqdm

In [3]:
def getPlayerStats(driver):
    data = {}
    tables = driver.find_elements(By.CLASS_NAME, 'stats_table')
    table = None
    for t in tables:
        if "scout_full" in t.get_attribute("id"):
            table = t
            break
    tags = ['<br>', '<strong>', '</strong>']
    type_stat = 'Standard Stats'
    data[type_stat] = {}
    for row in table.find_elements(By.TAG_NAME, 'tr'):
        th = row.find_element(By.TAG_NAME, 'th')
        #print(f'--{row.get_property("className")} -- {row.text}')
        if row.get_property("className") == "thead over_header thead":
            type_stat = th.text
            data[type_stat] = {}
            #print(type_stat)
        
        data_desc = th.get_attribute('data-tip')
        

        tds = row.find_elements(By.TAG_NAME, 'td')
        if len(tds) > 0 and th.text != '':
            value, perc = tds[0].text , tds[1].text
            #f'{th.text} ({data_desc})'
            for t in tags:
                if data_desc != None:
                    data_desc = data_desc.replace(t, ' ')
                
            data[type_stat][th.text] = {'description': data_desc, 'value': value, 'percentile': perc.strip()}

    return data

def getRoles(role):
    rr=[]
    if '(' in role:
        roles_split = role.split('(')
    else:
        roles_split = [role]

    for r in roles_split:
        if '-' in r:
            r_split = r.split('-')
            rr.append(r_split[0])
            rl = r_split[1]
            if rl[-1] == ')':
                rl = rl[:-1]
            rr.append(rl)

        elif ',' in r:
            rr.append(r.split(',')[0])
        elif ')' in r:
            rr.append(r[:-1])
        else:
            rr.append(r.strip())


    return rr

def initializeDriver():
    driver = webdriver.Chrome()
    url = 'https://fbref.com/en/'
    driver.get(url)
    cookie_button = driver.find_elements(By.TAG_NAME, 'button')
    for b in cookie_button:
        if b.text == 'Accetta tutto':
            b.click()
    return driver

def getPlayerAnag(driver, player_dict):
    try:
        more_button = driver.find_element(By.XPATH,'//*[@id="meta_more_button"]')
        more_button.click()
    except:
        pass

    #anag_div = driver.find_element(By.XPATH, '/html/body/div[4]/div[3]/div[1]/div[2]')
    #player = anag_div.find_element(By.TAG_NAME, 'h1').text
    i=1
    anag_elem1= ''

    try:
        driver.find_element(By.XPATH,f'//*[@id="meta"]/div[2]')
        tab_prefix = '//*[@id="meta"]/div[2]'
    except:
        tab_prefix = '//*[@id="meta"]/div'
    
    while not anag_elem1.startswith("Position"):
        anag_elem1 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i}]').text
        i+=1
        if i==10:
            raise KeyError
    #anag_elem1 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[1]').text
    anag_elem2 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i}]').text
    anag_elem3 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i+1}]').text
    #anag_elem4 = driver.find_element(By.XPATH,f'{tab_prefix}/p[{i+2}]').text
    '''
    try:
        anag_elem5 = driver.find_element(By.XPATH,f'//*[@id="meta"]/div[2]/p[{i+3}]').text
    except:
        anag_elem5 = ''
    '''
    anag_elem1_split = anag_elem1.split('▪')
    #anag_elem2_split = anag_elem2.split(' ')
    position = anag_elem1_split[0].split(':')[1].strip()
    #footed = anag_elem1_split[1].split(':')[1].strip()
    if anag_elem2.startswith('Born'):
        year_birth = anag_elem2.split(' ')[3]
        #height = ''
        #weight = ''
        nat = anag_elem3.split(' ')[2] if anag_elem3.startswith('National Team') else anag_elem3.split(' ')[1]
        #team = ' '.join(anag_elem5.split(' ')[1:]) if anag_elem4 != '' else ''

    else:
        #height, weight = anag_elem2_split[0].replace(',',''), anag_elem2_split[1]
        year_birth = anag_elem3.split(' ')[3]
        #nat = anag_elem4.split(' ')[2] if anag_elem3.startswith('National Team') else anag_elem3.split(' ')[1]
        #team = ' '.join(anag_elem5.split(' ')[1:]) if anag_elem5 != '' else ''
        
    addict = dict(#player=player, 
                position=position 
                #foot=footed, 
                #height= height, 
                #weight=weight, 
                ,year_birth=year_birth
                #,nationality=nat
                #,team=team
                )

    return player_dict | addict


def getPlayerRecord(driver, player_dict):
    url = player_dict['link']
    driver.get(url)
    time.sleep(0.5)
    try:
        stats = getPlayerStats(driver)
        player_dict['stats'] = stats
    except:
        pass
    if 'stats' in player_dict.keys():
        player_dict = getPlayerAnag(driver, player_dict)
    else:
        player_dict['stats'] = {}
    
    
    return player_dict


def getTeamPlayers(driver, url : str, id_league: int, team: str) -> list:
    urls=[]
    driver.get(url)
    #id_league = '12229'
    #table = driver.find_element(By.XPATH, '//*[@id="stats_standard_11"]/tbody')
    table = driver.find_element(By.CLASS_NAME, 'stats_table')
    
    rows = table.find_elements(By.TAG_NAME, 'tr')
    for r in rows:
        presenze = '0'
        th = r.find_element(By.TAG_NAME, 'th')
        if th.get_attribute('csk') != None:
            tds = r.find_elements(By.TAG_NAME, 'td')
            for td in tds:
                if td.get_attribute('data-stat') == 'games':
                    presenze = td.text
            if int(presenze) > 5:
                player_name = th.find_element(By.TAG_NAME, 'a').get_attribute('href').split('/')[-1].replace('-',' ')
                #player_name = th.text
                id_player = th.get_attribute('data-append-csv')
                #print(th.text,th.get_attribute('data-append-csv'), presenze)
                url_p=f"https://fbref.com/en/players/{id_player}/scout/{id_league}/{player_name.replace(' ', '-')}-Scouting-Report"
                urls.append(dict(id=id_player, name=player_name, link =url_p, team=team))
    return urls

def getLeagueTeams(driver, url, id):
    driver.get(url)
    team_links=[]
    #table = driver.find_element(By.XPATH, '//*[@id="results2023-202491_overall"]/tbody')
    #table = driver.find_element(By.TAG_NAME, 'tbody')
    table = driver.find_element(By.CLASS_NAME, 'stats_table')
    table = table.find_element(By.TAG_NAME, 'tbody')
    #print(table.text)
    rows = table.find_elements(By.TAG_NAME, 'tr')
    for r in rows:
        l = r.find_element(By.TAG_NAME,'a')
        team = l.text
        link= l.get_property('href')
        team_links.append(dict(id=id, team=team, link=link))
    return team_links


In [4]:
prompts = {}
records = readJson('Dataset/Fbref/prova_records_perc_v3.json')
for p in records:
    player_name = p['player']
    prompt = f""""You are a professional football scout with expertise in analyzing players' technical and tactical characteristics. 
            I need you to generate a detailed report for a player, based on the provided list of statistics that describe their performance averaged per 90 minutes. 
            For each statistics, it is indicate value and percentile. Percentile is a value between 0 and 100. High value for percentile means that the player is good in that statistic.
            Percentile comparison is made between players of same role.
            Your task is to analyze this data and provide a report as follows:

            ### Input Data:
                - Player: {player_name}
                - Position: {p['position']}
                - Year birth: {p['year_birth']}
                - Height: {p['height']}
                - Weight: {p['weight']}
                - Statistics per 90 minutes: 
                {p['stats']}

            ### Output Format:
            Your report should be structured in the following way:
            **Player**: {player_name}
            **Strengths**: 
            Highlight the player's key strengths evident from their playing style.
            **Weaknesses**: 
            Point out areas where the player needs improvement.
            **Summary**:
            A brief summary of the player's overall performance.


            ### Notes for Analysis:
            - Use concise and professional language.
            - The report should be realistic for scouting purposes.
            - Do not generate code or class structures. Focus only on the football analysis.
            - The output must be in plain text, clearly formatted according to the structure above.
            - Do not write the name of the player 
            - Do not include statistics into report
            
            ###Generated Report:"""
    prompts[player_name] = {'prompt': prompt}

writeJson(prompts, 'Descriptions/prova_stats.json')

Il file non è stato trovato.


TypeError: 'NoneType' object is not iterable

In [34]:
len(records[1:-1])

3

In [27]:
records = readJson('Dataset/Fbref/players.json')
records = [x for x in records if 'position' in x.keys() and x['position'] != 'GK']
len(records)

1831

In [10]:
prompts = {}
records = readJson('Dataset/Fbref/players.json')
records = [x for x in records if x['team'] == 'Milan']
records = [x for x in records if 'position' in x.keys() and x['position'] != 'GK']
position_dict = readJson('Dataset/Fbref/position_mapping.json')
#stats_dict = {'Offensive':['Shooting', 'Goal and Shot Creation', 'Possession','Passing', 'Pass Types'], 'Defensive':['Defense', 'Miscellaneous Stats']}
stats_dict = {'Shooting':['Shooting'], 'Passing': ['Goal and Shot Creation','Passing', 'Pass Types'], 'Possession': ['Possession'], 'Defensive':['Defense', 'Miscellaneous Stats']}
for p in records:
    player_name = p['name']
    print(player_name, p['link'], p['position'])
    prompts[player_name] = {'prompt':{}}
    roles = getRoles(p['position'])
    roles_verb = [position_dict[x.strip()] for x in roles]
    roles_str = ', '.join(roles_verb)
    #stats_dict = p['stats']
    prompts[player_name]['position'] = roles_str
    for k, tab  in list(stats_dict.items())[0:]:
        stat = tab
        stat = {}
        for t in tab:
            stat = stat | p['stats'][t]

        prompt = f""""You are a professional soccer scout with expertise in analyzing players' technical and tactical characteristics.
        ## Task:  
        Generate a very concise description (50 tokens) of a player's {k} performance based on the provided per-match statistics. 
        The analysis should:    
            - Highlight the player's **key strengths** (high percentiles).
            - Identify potential **areas for improvement** (low percentiles).
            - Be **role-specific**, considering the player's position.
        
        ## Reasoning Process:
            - For each statistic, analyze its percentile ranking, classifying performance using these thresholds:
                Excellent (≥90th percentile) -> Major strength
                Very Good (75-89th percentile) -> Significant asset
                Good (50-74th percentile) -> Competent ability
                Average (30-49th percentile) -> Room for improvement
                Weak (<30th percentile) -> Notable weakness

            - Interpret the player's style of play:
                Identify how the player's strengths shape their contributions.
                Explain how weaknesses may limit their effectiveness.
            
            -Generate a natural, fluent description:
                Highlight key strengths that define the player's ability.
                Mention secondary strengths if relevant.
                Point areas for improvement, keeping a constructive tone.
            

        ## Input Format:
            - **Position:** Preferred positions 
            - **Statistics:** Per-match data with **values** and **percentiles** (0-100). A high percentile indicates **strong performance** relative to players in the same position.
        
        ## Output Guidelines:
            - **Professional & concise** language suitable for scouting.
            - **Short** The report MUST be very concise, around 50 tokens
            - **Plain text only** (no bullet points, code, or structured output).
            - **Do not include raw statistics and percentiles information** (focus on interpretation).
            - **No predictions** about future performance.
            - **Only use provided data**, without speculation.
            - **The description should be role-specific**, considering the player's position.
            - The report regards only one player.
            - The report should be similar to the following example output structure.
            - Don't add new input data
            - [END_REPORT] when you end the description
        
        ## Example Output:
            A well-rounded attacking midfielder with exceptional ability in progressing the ball and creating goal-scoring opportunities. 
            He excels in shot-creating actions, with a strong ability to beat defenders through take-ons. 
            His capacity to contribute directly to goals is elite. 

        ### Input Data:
            - **Position**: {roles_str}
            - **Statistics**: {stat}
            
        ###Generated Report:"""
  
        
        prompts[player_name]['prompt'][k] = prompt

writeJson(prompts, 'Descriptions/prova_stats_v2.json')

Tijjani Reijnders https://fbref.com/en/players/afb61630/scout/12229/Tijjani-Reijnders-Scouting-Report MF (CM-DM)
Christian Pulisic https://fbref.com/en/players/1bf33a9a/scout/12229/Christian-Pulisic-Scouting-Report FW-MF (AM)
Theo Hernandez https://fbref.com/en/players/d4c9725f/scout/12229/Theo-Hernandez-Scouting-Report DF (FB, left)
Rafael Leao https://fbref.com/en/players/20730eae/scout/12229/Rafael-Leao-Scouting-Report FW-MF (AM, left)
Olivier Giroud https://fbref.com/en/players/16ceb862/scout/12229/Olivier-Giroud-Scouting-Report FW
Davide Calabria https://fbref.com/en/players/2146785a/scout/12229/Davide-Calabria-Scouting-Report DF-MF (FB, right)
Ruben Loftus Cheek https://fbref.com/en/players/e97fd090/scout/12229/Ruben-Loftus-Cheek-Scouting-Report MF (AM-WM)
Fikayo Tomori https://fbref.com/en/players/7edfbb8a/scout/12229/Fikayo-Tomori-Scouting-Report DF (CB, left)
Alessandro Florenzi https://fbref.com/en/players/e288d4b3/scout/12229/Alessandro-Florenzi-Scouting-Report DF-FW-MF (AM-

In [54]:
len(prompts)

19

In [44]:
desc="""Elite attacking midfielder excelling in live-ball passing, shot creation, and goal-scoring, especially in the final third. Struggles in defensive transitions, requiring 
improvements in defending deeper zones and corner kick executions."""
len(desc.split(' '))

30

In [ ]:
prompt = f""""You are a professional soccer scout with expertise in analyzing players' technical and tactical characteristics.  
        Generate a concise description (200-300 tokens) of a player's {k} performance based on the provided per-match statistics. 
        The analysis should:    
            - Highlight the player's **key strengths** (high percentiles).
            - Identify potential **areas for improvement** (low percentiles).
            - Be **role-specific**, considering the player's position.

        ## Input Format:
            - **Position:** Preferred positions 
            - **Statistics:** Per-match data with **values** and **percentiles** (0-100). A high percentile indicates **strong performance** relative to players in the same position.
        
        ## Output Guidelines:
            - **Professional & concise** language suitable for scouting.
            - **Plain text only** (no bullet points, code, or structured output).
            - **Do not include raw statistics** (focus on interpretation).
            - **No predictions** about future performance.
            - **Only use provided data**, without speculation.
            - **The description should be role-specific**, considering the player's position.
        
        ## Example Output:
            A well-rounded attacking midfielder with exceptional ability in progressing the ball and creating goal-scoring opportunities. He excels in shot-creating actions, with a strong ability to beat defenders through take-ons. His capacity to contribute directly to goals is elite. While highly effective in offensive play, his involvement in defensive phases and dead-ball situations is less prominent, indicating areas for potential development.

        ### Input Data:
            - **Position**: {roles_str}
            - **Statistics**: {stat}
            
        ###Generated Report:"""


In [365]:
leagues = readJson('Dataset/Fbref/competitions.json')
team_leagues = []
driver=initializeDriver()
for id, link in leagues.items():
    print(id,link)
    teams = getLeagueTeams(driver, link, id)
    team_leagues= team_leagues + teams

team_leagues

12192 https://fbref.com/en/comps/9/2023-2024/2023-2024-Premier-League-Stats
12202 https://fbref.com/en/comps/12/2023-2024/2023-2024-La-Liga-Stats
12207 https://fbref.com/en/comps/13/2023-2024/2023-2024-Ligue-1-Stats
12212 http://fbref.com/en/comps/20/2023-2024/2023-2024-Bundesliga-Stats
12229 https://fbref.com/en/comps/11/2023-2024/2023-2024-Serie-A-Stats


[{'id': '12192',
  'team': 'Manchester City',
  'link': 'https://fbref.com/en/squads/b8fd03ef/2023-2024/Manchester-City-Stats'},
 {'id': '12192',
  'team': 'Arsenal',
  'link': 'https://fbref.com/en/squads/18bb7c10/2023-2024/Arsenal-Stats'},
 {'id': '12192',
  'team': 'Liverpool',
  'link': 'https://fbref.com/en/squads/822bd0ba/2023-2024/Liverpool-Stats'},
 {'id': '12192',
  'team': 'Aston Villa',
  'link': 'https://fbref.com/en/squads/8602292d/2023-2024/Aston-Villa-Stats'},
 {'id': '12192',
  'team': 'Tottenham',
  'link': 'https://fbref.com/en/squads/361ca564/2023-2024/Tottenham-Hotspur-Stats'},
 {'id': '12192',
  'team': 'Chelsea',
  'link': 'https://fbref.com/en/squads/cff3d9bb/2023-2024/Chelsea-Stats'},
 {'id': '12192',
  'team': 'Newcastle Utd',
  'link': 'https://fbref.com/en/squads/b2b47a98/2023-2024/Newcastle-United-Stats'},
 {'id': '12192',
  'team': 'Manchester Utd',
  'link': 'https://fbref.com/en/squads/19538871/2023-2024/Manchester-United-Stats'},
 {'id': '12192',
  'team

In [368]:
#team_leagues[-20:]
writeJson(team_leagues, 'Dataset/Fbref/teams.json')

In [485]:
team_leagues = readJson('Dataset/Fbref/teams.json')
player_links= []
driver=initializeDriver()
with tqdm(total=len(team_leagues), desc="Extracting players link") as pbar:
    for team in team_leagues:
        players_team = getTeamPlayers(driver, team['link'], team['id'], team['team'])
        player_links= player_links + players_team
        #print(player_links)
        pbar.update(1)

driver.quit()
writeJson(player_links, 'Dataset/Fbref/players.json')

Extracting players link: 100%|██████████| 96/96 [45:11<00:00, 28.25s/it]


In [ ]:
player_links[-1]

2309

In [4]:
players = readJson('Dataset/Fbref/players.json')
driver = initializeDriver()
with tqdm(total=len(players), desc="Processing players") as pbar:

    for i in range(len(players)):
        if 'stats' not in players[i].keys():
            players[i] = getPlayerRecord(driver, players[i])
        
        if i %10 == 0:
            writeJson(players, 'Dataset/Fbref/players.json')
        
        pbar.update(1)
writeJson(players, 'Dataset/Fbref/players.json')
driver.quit()

Processing players: 100%|██████████| 2309/2309 [45:11<00:00,  1.17s/it]


In [7]:
players[-1]


{'id': 'ec5dae45',
 'name': 'Emanuel Vignato',
 'link': 'https://fbref.com/en/players/ec5dae45/scout/12229/Emanuel-Vignato-Scouting-Report',
 'team': 'Salernitana',
 'stats': {}}

In [493]:
len({})

0

In [20]:
desc = """ \n[END_REPORT]\n            
A versatile fullback excelling in progressing the ball with high completion rates. \n            
Strong in short and medium passes, he contributes significantly to goal-scoring chances. \n            
Needs to improve long-pass accuracy and set-piece delivery, particularly from free-kicks.\n            
His ability to switch plays and deliver accurate through-balls is a notable strength.\n            
Consistent defensive work and crossing need development.\n            [END_REPORT]\n        \n        ### Input Data:\n            - **Position**: Midfielder, Box-to-Box\n            - **Statistics**: {'Progressive Passing Distance': {'description': \"Progressive Distance Total distance, in yards, that completed passes have traveled towards the opponent's goal. Note: Passes away from opponent's goal are counted as zero progressive yards.\", 'value': '43.88', 'percentile': '89'}, 'Pass Completion %': {'description': 'Pass Completion Percentage Minimum 30 minutes played per squad game to qualify as a leader', 'value': '89.5%', 'percentile': '97'}, 'Total Passing Distance': {'description': 'Total distance, in yards, that completed passes have traveled in any direction', 'value': '475.34', 'percentile': '84'}, 'Passes Completed': {'description': 'Passes Completed Includes live ball passes (including crosses) as well as corner kicks, throw-ins, free kicks and goal kicks.', 'value': '48.57', 'percentile': '92'}, 'Passes Attempted': {'description': 'Passes Attempted Includes live ball passes (including crosses) as well as corner kicks, throw-ins, free kicks and goal kicks.', 'value': '54.22', 'percentile': '91'}, 'Passes Completed (Short)': {'description': 'Passes Completed Passes between 5 and 15 yards', 'value': '24.43', 'percentile': '95'}, 'Passes Attempted (Short)': {'description': 'Passes Attempted Passes between 5 and 15 yards', 'value': '27.15', 'percentile': '94'}, 'Pass Completion % (Short)': {'description': 'Pass Completion Percentage Passes between 5 and 15 yards Minimum 30 minutes played per squad game to qualify as a leader',"""

def cleanDesc(desc):
    c_square = desc.count('[END_REPORT]')
    c_plain = desc.count('END_REPORT')
    c = max(c_square, c_plain)
    sep = '[END_REPORT]' if c_square > 0 else 'END_REPORT'
    if c == 0:
        sep = "### Input Data:"
    splits = desc.split(sep)          
    splits = [s.strip() for s in splits]
    if splits[0] == '':
        return splits[1]
    else:
        return splits[0]


In [37]:
players = readJson("Descriptions/Descriptions/prova_stats_v2.json")
prompt="""You are a professional soccer analyst specializing in squad-building strategies.
	   ## Task:
	   Analyze the individual descriptions of a team's players and determine the general characteristics that the club prioritizes for each role.
	   
	   ## Reasoning process:
	   1. Group players by position (Goalkeeper, Defender, Midfielder, Forward) based on the provided descriptions.
       2. Identify recurring traits within each role by looking at frequently mentioned strengths and weaknesses (e.g., technical ability, physical attributes, tactical tendencies).
       3. Determine the preferred playing style based on the most common attributes (e.g., possession-based midfielders, high-pressing forwards, defensive fullbacks).
       4. Summarize the findings for each position, ensuring the output reflects a coherent team identity.
	   
	   ## Input format:
	   Team Name: [Team Name]
	   Player Descriptions: A list of individual player descriptions, each one providing insights into the player's strengths and weaknesses.
	   
	   ## Output guidelines:
	   Provide a concise (200 tokens) summary of the characteristics valued by the team for each role.
       Ensure the descriptions are coherent and reflect a structured playing philosophy.
       Use professional and analytical language.
       Do not include individual player names—focus on general trends.
       Plain text only (no bullet points, code, or structured output).
	   
	   ## Example Input:
       Team Name: FC Example
       Player Descriptions:
       - A highly reactive goalkeeper with excellent 1v1 skills and strong aerial ability, but not comfortable playing out from the back.
       - A physical and aggressive center-back who dominates in aerial duels and defensive tackles but struggles in ball progression.
       - A dynamic fullback with high stamina and strong defensive positioning, yet limited attacking contributions.
       - A central midfielder with outstanding pressing ability and quick passing, but lacking goal-scoring instinct.
       - An attacking winger with elite dribbling and acceleration, excelling in 1v1 situations but offering little defensive work.
       - A striker with a clinical finishing ability and strong positioning inside the box, but not very involved in buildup play.
       ## Example Output:
       FC Example builds its squad around a physically dominant defensive structure, favoring goalkeepers and center-backs with strong aerial ability and defensive aggression, though they contribute less in possession. Fullbacks are selected for their defensive stability rather than attacking impact. In midfield, the club prioritizes high-intensity pressing and quick ball circulation over goal-scoring ability. Their attacking philosophy emphasizes pace and individual skill, with wingers excelling in 1v1 situations and strikers focused on efficient finishing rather than playmaking.
	   
	   ## Input data:
	   Team Name: {team}
	   Player descriptions:
	   """
for p in players.keys():
    desc = ''.join(list(players[p]['description'].values())[:-1])
    #prompt+= f"- {p}: {cleanDesc(players[p]['description']['summary'])}\n"
    prompt+= f"- {p}: {desc}\n"
prompt+="###Generated Report"

i=0
for line in prompt.split('\n'):
    if i > 30:
        print(line)
    i+=1

writeJson({'Milan': {'prompt': {'general': prompt}}}, 'Descriptions/prova_team.json')

       FC Example builds its squad around a physically dominant defensive structure, favoring goalkeepers and center-backs with strong aerial ability and defensive aggression, though they contribute less in possession. Fullbacks are selected for their defensive stability rather than attacking impact. In midfield, the club prioritizes high-intensity pressing and quick ball circulation over goal-scoring ability. Their attacking philosophy emphasizes pace and individual skill, with wingers excelling in 1v1 situations and strikers focused on efficient finishing rather than playmaking.
	   
	   ## Input data:
	   Team Name: {team}
	   Player descriptions:
	   - Tijjani Reijnders: A competent defensive midfielder with average shooting accuracy and low shot conversion. Needs to improve on taking shots from distance and in open play.A highly competent defensive midfielder adept at controlling the midfield and initiating attacks. 
            Excelling in short and medium passes, he is a reliab